# 验证数据一致性

验证 `data/pressure/train/pressure.npy` 中索引为 2251 的样本，是否与 `tests/AT_A_1.h5` 中的原始压力数据完全一致。

验证分为三步：

1. 通过 `manifest.csv` 确认索引 2251 对应的样本确实是 `AT_A_1`；
2. 分别读取 NPY 样本和 HDF5 中的 `pressure_data`；
3. 检查形状、数据类型和每个元素是否一致。

In [1]:
# 1. 导入依赖并定位项目目录
from pathlib import Path
import csv
import os

import h5py
import numpy as np


def find_project_root(start=None):
    """从当前目录向上查找项目根目录。"""
    override = os.environ.get("STEMNIST_PROJECT_ROOT")
    current = Path(override).expanduser() if override else Path(start or Path.cwd())
    current = current.resolve()

    for candidate in (current, *current.parents):
        if (candidate / "data" / "pressure").is_dir():
            return candidate

    raise FileNotFoundError(
        "未找到项目根目录。请在项目目录内运行此 notebook，"
        "或设置 STEMNIST_PROJECT_ROOT 环境变量。"
    )


PROJECT_ROOT = find_project_root()
ROW_INDEX = 2251
SAMPLE_ID = "AT_A_1"

NPY_PATH = PROJECT_ROOT / "data" / "pressure" / "train" / "pressure.npy"
MANIFEST_PATH = PROJECT_ROOT / "data" / "pressure" / "train" / "manifest.csv"
H5_PATH = PROJECT_ROOT / "tests" / f"{SAMPLE_ID}.h5"
H5_DATASET_NAME = "pressure_data"

required_paths = (NPY_PATH, MANIFEST_PATH, H5_PATH)
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"缺少验证所需文件：{missing_paths}")

print(f"项目目录：{PROJECT_ROOT}")
print(f"待验证样本：train[{ROW_INDEX}] <-> {H5_PATH.name}")

项目目录：C:\Users\Fortyfour\Desktop\STEMNIST_Classify
待验证样本：train[2251] <-> AT_A_1.h5


In [2]:
# 2. 核对 manifest，避免只凭固定索引比较错误的样本
def read_manifest_row(manifest_path, row_index):
    """按 CSV 中的物理行号读取一条样本记录。"""
    with Path(manifest_path).open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        for position, row in enumerate(reader):
            if position == row_index:
                return row

    raise IndexError(f"manifest 中不存在索引 {row_index}")


manifest_row = read_manifest_row(MANIFEST_PATH, ROW_INDEX)
manifest_index = int(manifest_row["row_index"])

if manifest_index != ROW_INDEX:
    raise AssertionError(
        f"manifest 行号不一致：期望 {ROW_INDEX}，实际 {manifest_index}"
    )
if manifest_row["sample_id"] != SAMPLE_ID:
    raise AssertionError(
        f"索引 {ROW_INDEX} 对应 {manifest_row['sample_id']}，而不是 {SAMPLE_ID}"
    )

print("manifest 样本信息：")
print(f"  row_index     = {manifest_row['row_index']}")
print(f"  sample_id     = {manifest_row['sample_id']}")
print(f"  label         = {manifest_row['label']}")
print(f"  source_member = {manifest_row['source_member']}")

manifest 样本信息：
  row_index     = 2251
  sample_id     = AT_A_1
  label         = A
  source_member = STEMNIST Dataset/RawCharacters/AT_A_1.h5


In [3]:
# 3. 读取 NPY 中的目标样本和 HDF5 中的原始数据
pressure = np.load(NPY_PATH, mmap_mode="r", allow_pickle=False)
if pressure.ndim != 4:
    raise ValueError(f"pressure.npy 应为四维数组，实际形状为 {pressure.shape}")
if not 0 <= ROW_INDEX < len(pressure):
    raise IndexError(f"索引 {ROW_INDEX} 超出 pressure.npy 的样本范围")

# copy() 使目标样本不再依赖内存映射文件。
npy_sample = np.asarray(pressure[ROW_INDEX]).copy()
del pressure

with h5py.File(H5_PATH, "r") as h5_file:
    if H5_DATASET_NAME not in h5_file:
        raise KeyError(
            f"{H5_PATH.name} 中缺少数据集 {H5_DATASET_NAME!r}；"
            f"现有顶层对象：{list(h5_file.keys())}"
        )
    h5_sample = h5_file[H5_DATASET_NAME][...]

print(f"NPY 样本：shape={npy_sample.shape}, dtype={npy_sample.dtype}")
print(f"HDF5 数据：shape={h5_sample.shape}, dtype={h5_sample.dtype}")

NPY 样本：shape=(240, 16, 16), dtype=uint8
HDF5 数据：shape=(240, 16, 16), dtype=uint8


In [4]:
# 4. 比较形状、数据类型和所有元素
same_shape = npy_sample.shape == h5_sample.shape
same_dtype = npy_sample.dtype == h5_sample.dtype
same_values = np.array_equal(npy_sample, h5_sample)

if same_shape:
    difference = npy_sample.astype(np.int16) - h5_sample.astype(np.int16)
    mismatch_count = int(np.count_nonzero(difference))
    max_absolute_error = int(np.max(np.abs(difference)))
else:
    mismatch_count = None
    max_absolute_error = None

print("验证结果：")
print(f"  形状一致：{same_shape}")
print(f"  类型一致：{same_dtype}")
print(f"  数值一致：{same_values}")
print(f"  不一致元素数：{mismatch_count}")
print(f"  最大绝对误差：{max_absolute_error}")

if not (same_shape and same_dtype and same_values):
    raise AssertionError("数据一致性验证失败，请检查上面的差异统计。")

print(f"验证通过：pressure.npy[{ROW_INDEX}] 与 {H5_PATH.name} 完全一致。")

验证结果：
  形状一致：True
  类型一致：True
  数值一致：True
  不一致元素数：0
  最大绝对误差：0
验证通过：pressure.npy[2251] 与 AT_A_1.h5 完全一致。
